# LLM Pipeline Profiler — Kaggle Notebook Example

This notebook demonstrates how to instrument a HuggingFace LLM inference pipeline using `llm-profiler`.

It showcases two main styles of profiling:
1. **Style 1: Auto-Instrumentation (Default / Zero-Code Changes)**: Profile a standard HuggingFace pipeline automatically by simply initializing the `Tracer` class.
2. **Style 2: Manual Staging**: Use the `tracer.stage(...)` context manager to profile a custom autoregressive loop or custom non-HuggingFace components with fine-grained control.

In [ ]:
# 1. Install the profiler library
# NOTE: For local testing/development of auto-instrumentation, install the local version:
# !pip install -e ../profiler-lib
# 
# For remote Kaggle notebook usage, install directly from GitHub:
!pip install git+https://github.com/harshgupta-23/llm-pipeline-profiler.git#subdirectory=profiler-lib

In [ ]:
# 2. Common imports and configuration
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm_profiler import Tracer

# Auto-detect device (GPU/CUDA if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running LLM generation profile on device: {device}")

## Style 1: Auto-Instrumentation (Zero-Code Changes)

In auto-instrumentation mode (enabled by default), `llm-profiler` automatically patches HuggingFace's key entry points:
- `AutoModelForCausalLM.from_pretrained` (and base `PreTrainedModel.from_pretrained`) -> stage name `"model_load"`
- `PreTrainedTokenizerBase.__call__` -> stage name `"tokenize"`
- `GenerationMixin.generate` -> stage name `"generate"`
- `PreTrainedTokenizerBase.decode` / `batch_decode` -> stage name `"postprocess"`

This allows you to profile standard pipelines with zero code changes.

In [ ]:
# Initialize Tracer (auto_instrument is True by default)
# Replace 'dashboard_url' with your actual ngrok forwarding URL (e.g. "https://xxxx.ngrok-free.dev") for live streaming,
# or set to None to save results locally in /kaggle/working/run.json
tracer_auto = Tracer(
    run_name="kaggle-distilgpt2-auto-run",
    model_name="distilgpt2",
    dashboard_url=None,
    profile_torch=True
)

# Run typical HF pipeline with ZERO code changes!
print("--- Auto-instrumented Pipeline Start ---")
model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
inputs = tokenizer("The quick brown fox", return_tensors="pt").to(device)
output = model.generate(**inputs, max_new_tokens=40)
text = tokenizer.decode(output[0])
print("--- Generated Output (Auto) ---")
print(text)
print("-------------------------------\n")

# Export results. This stops the auto-instrumentation tracing as well.
tracer_auto.export()
print("Auto-instrumentation run exported.")

## Style 2: Manual Staging

For custom models, non-HuggingFace pipelines, or if you need custom stage boundaries and custom logged metrics, you can explicitly disable auto-instrumentation and use the `tracer.stage(...)` context manager.

In [ ]:
# Initialize Tracer with auto_instrument=False
tracer_manual = Tracer(
    run_name="kaggle-distilgpt2-manual-run",
    model_name="distilgpt2",
    dashboard_url=None,
    auto_instrument=False
)

print("--- Manually Staged Pipeline Start ---")

# Stage 1: Load Model & Tokenizer
with tracer_manual.stage("model_load"):
    print("Loading model and tokenizer...")
    model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
    tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

# Stage 2: Tokenization
with tracer_manual.stage("tokenize"):
    prompt = "Deep learning and artificial intelligence are revolutionizing"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Stage 3: Custom Autoregressive Token Generation (with PyTorch trace & custom metric logging)
with tracer_manual.stage("generate", profile_torch=True):
    print("Starting generation loop...")
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    start_time = time.perf_counter()
    tokens_generated = 0
    max_new_tokens = 40

    for i in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            
            # Append generated token to inputs
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones((1, 1), device=device, dtype=torch.long)
            ], dim=-1)
            
        tokens_generated += 1
        elapsed = time.perf_counter() - start_time
        
        # Calculate tokens per second (throughput) and log it
        tps = tokens_generated / elapsed if elapsed > 0 else 0.0
        tracer_manual.log_metric("tps", tps)
        
        # Slight delay to mimic processing latency and allow metrics sampling
        time.sleep(0.02)

# Stage 4: Post-processing (Decoding)
with tracer_manual.stage("postprocess"):
    output_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

print("\n--- Generated Output (Manual) ---")
print(output_text)
print("---------------------------------\n")

# Export manual profiling data
tracer_manual.export()
print("Manual run exported successfully!")